# Neural scenario generation: synthetic pipeline test
Select GPU if available; CPU is adequate for this tiny test. Run all cells in order. The included 12 pairs are explicitly synthetic and are not DCSASS ground truth. Randomly initialized GRU weights, training-only vocabulary, held-out validation, and teacher forcing implement a neural pipeline. These few examples cannot establish investigative usefulness. Generated text may be incorrect or uncautious; human review is mandatory. No pretrained language model or external inference API is used.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/AI-Crime-Investigation-FYP')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ranaalihaider99993-source/AI-Crime-Investigation-FYP.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
from scripts.common import seed_all
seed_all(42)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/AI-Crime-Investigation-FYP/canonical-four-class-v2')
except ImportError:
    OUTPUT = REPO / 'artifacts'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Persistent output:', OUTPUT)


In [ ]:
from scripts.scenario import load_pairs, train, infer
PAIRS = REPO / 'annotations/synthetic_scenarios.csv'
rows = load_pairs(PAIRS)
print('Loaded pairs:', len(rows), 'provenance:', sorted({r['provenance'] for r in rows}))
# For real training, replace PAIRS with a sufficiently large human-reviewed paired CSV.
# See annotations/README.md. Do not invent real narratives from class labels.
history = train(PAIRS, OUTPUT / 'scenario', epochs=5)


In [ ]:
example = infer(OUTPUT / 'scenario/scenario_gru.pth',
                'classification normal observation person standing')
print(example)


Inputs currently serialize evidence as text. Actual detector output, classifier probabilities, and investigator observations must later be encoded into a consistent schema and annotated by reviewers. Synthetic loss/token accuracy only diagnose execution. Validation uses teacher forcing; it does not measure factual grounding. The best checkpoint and epoch history persist on Drive. A restarted training run starts from scratch; use the saved checkpoint for inference without retraining.